In [1]:
import os
import sys
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("pyarrow:", pa.__version__)
print("duckdb:", duckdb.__version__)

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("ICEBERG_SPARK_JAR:", os.environ.get("ICEBERG_SPARK_JAR"))

Python: 3.11.15 (main, May 29 2026, 05:29:19) [GCC 12.2.0]
pandas: 2.2.3
pyarrow: 16.1.0
duckdb: 1.1.3
JAVA_HOME: /opt/java/openjdk
SPARK_HOME: /home/airflow/.local/lib/python3.11/site-packages/pyspark
ICEBERG_SPARK_JAR: /home/airflow/.local/lib/python3.11/site-packages/pyspark/jars/iceberg-spark-runtime-3.5_2.12-1.10.0.jar


In [2]:
import pyspark

iceberg_jar = os.environ.get("ICEBERG_SPARK_JAR")

print("PySpark:", pyspark.__version__)
print("Iceberg JAR exists:", os.path.exists(iceberg_jar) if iceberg_jar else False)
print("Iceberg JAR path:", iceberg_jar)

PySpark: 3.5.6
Iceberg JAR exists: True
Iceberg JAR path: /home/airflow/.local/lib/python3.11/site-packages/pyspark/jars/iceberg-spark-runtime-3.5_2.12-1.10.0.jar


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

base_dir = Path("/opt/airflow/data/iceberg_hive_demo")
incoming_dir = base_dir / "incoming"
hive_dir = base_dir / "hive_events"
iceberg_warehouse = base_dir / "iceberg_warehouse"

incoming_dir.mkdir(parents=True, exist_ok=True)

csv_path = incoming_dir / "events_fake.csv"

# Make the demo deterministic.
# This means every student should get the same fake data each time.
rng = np.random.default_rng(seed=42)

timestamps = pd.date_range(
    start="2026-06-25 00:00:00",
    end="2026-06-26 23:55:00",
    freq="5min"
)

devices = np.array(["ios", "android", "web"])
countries = np.array(["US", "CA", "GB", "DE"])
pages = np.array(["/home", "/search", "/product", "/cart", "/checkout"])

df = pd.DataFrame({
    "event_time": timestamps,
    "user_id": rng.integers(1000, 1100, size=len(timestamps)),
    "amount": np.round(rng.uniform(5, 250, size=len(timestamps)), 2),
    "device": rng.choice(devices, size=len(timestamps)),
    "country": rng.choice(countries, size=len(timestamps)),
    "page_url": rng.choice(pages, size=len(timestamps)),
})

# Sort by event_time before writing the CSV.
# Later, when we write Parquet files, this sorted order will make
# the Parquet min/max statistics easier to understand.
df = df.sort_values("event_time").reset_index(drop=True)

df.to_csv(csv_path, index=False)

print("Wrote:", csv_path)
print("Rows:", len(df))

df.head(10)

Wrote: /opt/airflow/data/iceberg_hive_demo/incoming/events_fake.csv
Rows: 576


,event_time,user_id,amount,device,country,page_url
0,2026-06-25 00:00:00,1008,133.32,android,DE,/home
1,2026-06-25 00:05:00,1077,29.91,ios,US,/search
2,2026-06-25 00:10:00,1065,209.20,web,CA,/product
3,2026-06-25 00:15:00,1043,17.73,web,US,/home
4,2026-06-25 00:20:00,1043,231.59,web,CA,/search
5,2026-06-25 00:25:00,1085,29.28,web,GB,/home
6,2026-06-25 00:30:00,1008,211.68,web,CA,/checkout
7,2026-06-25 00:35:00,1069,226.15,web,US,/home
8,2026-06-25 00:40:00,1020,244.99,android,GB,/home
9,2026-06-25 00:45:00,1009,201.50,ios,US,/home


In [4]:
import shutil
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Remove the previous Hive demo table if it exists.
# This makes the notebook repeatable.
if hive_dir.exists():
    shutil.rmtree(hive_dir)

hive_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path, parse_dates=["event_time"])
df = df.sort_values("event_time").reset_index(drop=True)

def write_one_parquet_file(input_df: pd.DataFrame, output_path: Path):
    """
    Write one sorted Parquet file with small row groups.

    We intentionally keep row_group_size small for teaching.
    Real production files would usually have much larger row groups.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)

    table = pa.Table.from_pandas(input_df, preserve_index=False)

    pq.write_table(
        table,
        output_path,
        compression="snappy",
        row_group_size=24
    )

def write_hive_day_files(day_string: str, ranges: list[tuple[str, str]]):
    day_df = df[df["event_time"].dt.strftime("%Y-%m-%d") == day_string].copy()

    year = day_string[0:4]
    month = day_string[5:7]
    day = day_string[8:10]

    partition_dir = hive_dir / f"year={year}" / f"month={month}" / f"day={day}"

    for i, (start_time, end_time) in enumerate(ranges):
        start_ts = pd.Timestamp(f"{day_string} {start_time}")
        end_ts = pd.Timestamp(f"{day_string} {end_time}")

        part_df = day_df[
            (day_df["event_time"] >= start_ts)
            & (day_df["event_time"] <= end_ts)
        ].copy()

        part_df = part_df.sort_values("event_time").reset_index(drop=True)

        output_path = partition_dir / f"part-{i:03d}.parquet"
        write_one_parquet_file(part_df, output_path)

        print(
            output_path,
            "rows=",
            len(part_df),
            "min=",
            part_df["event_time"].min(),
            "max=",
            part_df["event_time"].max()
        )

write_hive_day_files(
    "2026-06-25",
    [
        ("00:00:00", "11:55:00"),
        ("12:00:00", "23:55:00"),
    ]
)

write_hive_day_files(
    "2026-06-26",
    [
        ("00:00:00", "07:55:00"),
        ("08:00:00", "15:55:00"),
        ("16:00:00", "23:55:00"),
    ]
)

/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=25/part-000.parquet rows= 144 min= 2026-06-25 00:00:00 max= 2026-06-25 11:55:00
/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=25/part-001.parquet rows= 144 min= 2026-06-25 12:00:00 max= 2026-06-25 23:55:00
/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-000.parquet rows= 96 min= 2026-06-26 00:00:00 max= 2026-06-26 07:55:00
/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-001.parquet rows= 96 min= 2026-06-26 08:00:00 max= 2026-06-26 15:55:00
/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-002.parquet rows= 96 min= 2026-06-26 16:00:00 max= 2026-06-26 23:55:00


In [5]:
from pathlib import Path
import pyarrow.parquet as pq

def inspect_parquet_file(path: Path, column_name: str = "event_time"):
    pf = pq.ParquetFile(path)

    print("=" * 100)
    print("File:", path)
    print("Num row groups:", pf.num_row_groups)
    print("Schema:")
    print(pf.schema)

    arrow_schema = pf.schema_arrow
    column_index = arrow_schema.get_field_index(column_name)

    print(f"\nRow-group statistics for column: {column_name}")

    for rg_idx in range(pf.num_row_groups):
        rg = pf.metadata.row_group(rg_idx)
        col = rg.column(column_index)
        stats = col.statistics

        print(
            f"  row_group_{rg_idx}:",
            "rows=", rg.num_rows,
            "min=", stats.min if stats else None,
            "max=", stats.max if stats else None,
        )

# Inspect the middle file for 2026-06-26.
hive_middle_file = hive_dir / "year=2026" / "month=06" / "day=26" / "part-001.parquet"
inspect_parquet_file(hive_middle_file)

File: /opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-001.parquet
Num row groups: 4
Schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 event_time (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 user_id;
  optional double field_id=-1 amount;
  optional binary field_id=-1 device (String);
  optional binary field_id=-1 country (String);
  optional binary field_id=-1 page_url (String);
}


Row-group statistics for column: event_time
  row_group_0: rows= 24 min= 2026-06-26 08:00:00 max= 2026-06-26 09:55:00
  row_group_1: rows= 24 min= 2026-06-26 10:00:00 max= 2026-06-26 11:55:00
  row_group_2: rows= 24 min= 2026-06-26 12:00:00 max= 2026-06-26 13:55:00
  row_group_3: rows= 24 min= 2026-06-26 14:00:00 max= 2026-06-26 15:55:00


In [6]:
def parquet_file_summary(path: Path, column_name: str = "event_time"):
    pf = pq.ParquetFile(path)
    arrow_schema = pf.schema_arrow
    column_index = arrow_schema.get_field_index(column_name)

    mins = []
    maxs = []
    rows = 0

    for rg_idx in range(pf.num_row_groups):
        rg = pf.metadata.row_group(rg_idx)
        rows += rg.num_rows

        col = rg.column(column_index)
        stats = col.statistics

        if stats is not None:
            mins.append(stats.min)
            maxs.append(stats.max)

    return {
        "file": str(path.relative_to(hive_dir)),
        "rows": rows,
        "row_groups": pf.num_row_groups,
        "event_time_min": min(mins),
        "event_time_max": max(maxs),
    }

hive_file_summaries = [
    parquet_file_summary(path)
    for path in sorted(hive_dir.rglob("*.parquet"))
]

pd.DataFrame(hive_file_summaries)

,file,rows,row_groups,event_time_min,event_time_max
0,year=2026/month=06/day=25/part-000.parquet,144,6,2026-06-25 00:00:00,2026-06-25 11:55:00
1,year=2026/month=06/day=25/part-001.parquet,144,6,2026-06-25 12:00:00,2026-06-25 23:55:00
2,year=2026/month=06/day=26/part-000.parquet,96,4,2026-06-26 00:00:00,2026-06-26 07:55:00
3,year=2026/month=06/day=26/part-001.parquet,96,4,2026-06-26 08:00:00,2026-06-26 15:55:00
4,year=2026/month=06/day=26/part-002.parquet,96,4,2026-06-26 16:00:00,2026-06-26 23:55:00


In [8]:
import duckdb

con = duckdb.connect()

hive_glob = str(hive_dir / "**" / "*.parquet")

query = f"""
SELECT user_id, amount
FROM read_parquet('{hive_glob}', hive_partitioning = true)
WHERE event_time >= TIMESTAMP '2026-06-26 10:15:00'
  AND event_time <  TIMESTAMP '2026-06-26 10:30:00'
ORDER BY event_time;
"""

result = con.execute(query).df()
result

,user_id,amount
0,1022,78.35
1,1027,64.14
2,1013,84.01


In [9]:
pd.read_parquet("../data/iceberg_hive_demo/hive_events/year=2026/month=06/day=25/part-001.parquet").head()

,event_time,user_id,amount,device,country,page_url
0,2026-06-25 12:00:00,1099,176.04,android,DE,/search
1,2026-06-25 12:05:00,1085,210.31,web,US,/cart
2,2026-06-25 12:10:00,1003,14.84,android,DE,/home
3,2026-06-25 12:15:00,1023,54.44,ios,DE,/cart
4,2026-06-25 12:20:00,1082,35.61,ios,US,/search


In [11]:
query_with_partitions = f"""
SELECT
    event_time,
    user_id,
    amount,
    year,
    month,
    day
FROM read_parquet('{hive_glob}', hive_partitioning = true)
WHERE event_time >= TIMESTAMP '2026-06-26 10:15:00'
  AND event_time <  TIMESTAMP '2026-06-26 10:30:00'
ORDER BY event_time;
"""

con.execute(query_with_partitions).df()

,event_time,user_id,amount,year,month,day
0,2026-06-26 10:15:00,1022,78.35,2026,06,26
1,2026-06-26 10:20:00,1027,64.14,2026,06,26
2,2026-06-26 10:25:00,1013,84.01,2026,06,26


In [16]:
explain_query = f"""
EXPLAIN
SELECT user_id, amount
FROM read_parquet('{hive_glob}', hive_partitioning = true)
WHERE event_time >= TIMESTAMP '2026-06-26 10:15:00'
  AND event_time <  TIMESTAMP '2026-06-26 10:30:00';
"""

plan_df = con.execute(explain_query).df()
pd.set_option("display.max_colwidth", None)
print(plan_df['explain_value'][0])

┌───────────────────────────┐
│       READ_PARQUET        │
│    ────────────────────   │
│         Function:         │
│        READ_PARQUET       │
│                           │
│        Projections:       │
│          user_id          │
│           amount          │
│                           │
│          Filters:         │
│ event_time>='2026-06-26 10│
│ :15:00'::TIMESTAMP_NS AND │
│  event_time<'2026-06-26 10│
│ :30:00'::TIMESTAMP_NS AND │
│   event_time IS NOT NULL  │
│                           │
│         ~144 Rows         │
└───────────────────────────┘



In [18]:
metadata_df = con.execute(f"""
SELECT *
FROM parquet_metadata('{hive_middle_file}')
""").df()

metadata_df.head()

,file_name,row_group_id,row_group_num_rows,row_group_num_columns,row_group_bytes,column_id,file_offset,num_values,path_in_schema,type,...,stats_min_value,stats_max_value,compression,encodings,index_page_offset,dictionary_page_offset,data_page_offset,total_compressed_size,total_uncompressed_size,key_value_metadata
0,/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-001.parquet,0,24,6,1167,0,286,24,event_time,INT64,...,2026-06-26 08:00:00,2026-06-26 09:55:00,SNAPPY,"PLAIN, RLE, RLE_DICTIONARY",NaN,4,200,282,292,{}
1,/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-001.parquet,0,24,6,1167,1,600,24,user_id,INT64,...,1003,1099,SNAPPY,"PLAIN, RLE, RLE_DICTIONARY",NaN,387,514,213,292,{}
2,/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-001.parquet,0,24,6,1167,2,963,24,amount,DOUBLE,...,9.33,249.41,SNAPPY,"PLAIN, RLE, RLE_DICTIONARY",NaN,699,877,264,292,{}
3,/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-001.parquet,0,24,6,1167,3,1152,24,device,BYTE_ARRAY,...,android,web,SNAPPY,"PLAIN, RLE, RLE_DICTIONARY",NaN,1061,1101,91,88,{}
4,/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-001.parquet,0,24,6,1167,4,1308,24,country,BYTE_ARRAY,...,CA,US,SNAPPY,"PLAIN, RLE, RLE_DICTIONARY",NaN,1224,1263,84,81,{}


In [19]:
useful_cols = [
    col for col in metadata_df.columns
    if col.lower() in {
        "file_name",
        "row_group_id",
        "row_group_num_rows",
        "path_in_schema",
        "stats_min",
        "stats_max",
        "total_compressed_size",
        "total_uncompressed_size",
    }
]

metadata_df[useful_cols].query("path_in_schema == 'event_time'")

,file_name,row_group_id,row_group_num_rows,path_in_schema,stats_min,stats_max,total_compressed_size,total_uncompressed_size
0,/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-001.parquet,0,24,event_time,2026-06-26 08:00:00,2026-06-26 09:55:00,282,292
6,/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-001.parquet,1,24,event_time,2026-06-26 10:00:00,2026-06-26 11:55:00,281,292
12,/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-001.parquet,2,24,event_time,2026-06-26 12:00:00,2026-06-26 13:55:00,283,292
18,/opt/airflow/data/iceberg_hive_demo/hive_events/year=2026/month=06/day=26/part-001.parquet,3,24,event_time,2026-06-26 14:00:00,2026-06-26 15:55:00,282,292


In [20]:
import os
from pyspark.sql import SparkSession

iceberg_jar = os.environ["ICEBERG_SPARK_JAR"]

spark = (
    SparkSession.builder
    .appName("hive-vs-iceberg-class-demo")
    .master("local[*]")
    .config("spark.jars", iceberg_jar)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", str(iceberg_warehouse))
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Iceberg warehouse:", iceberg_warehouse)

26/06/26 22:41:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark: 3.5.6
Iceberg warehouse: /opt/airflow/data/iceberg_hive_demo/iceberg_warehouse


In [21]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.demo")

spark.sql("""
CREATE TABLE IF NOT EXISTS local.demo.test_table (
    id INT,
    name STRING
)
USING iceberg
""")

spark.sql("""
INSERT INTO local.demo.test_table VALUES
(1, 'Alice'),
(2, 'Bob')
""")

spark.sql("""
SELECT * FROM local.demo.test_table
""").show()

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



In [23]:
import shutil

spark.sql("DROP TABLE IF EXISTS local.demo.test_table")
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.demo")

table_path = iceberg_warehouse / "demo" / "test_table"

if table_path.exists():
    shutil.rmtree(table_path)

print("Cleaned:", table_path)

Cleaned: /opt/airflow/data/iceberg_hive_demo/iceberg_warehouse/demo/test_table


In [24]:
events_sdf = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(csv_path))
)

events_sdf.printSchema()
events_sdf.show(5, truncate=False)

root
 |-- event_time: timestamp (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- page_url: string (nullable = true)

+-------------------+-------+------+-------+-------+--------+
|event_time         |user_id|amount|device |country|page_url|
+-------------------+-------+------+-------+-------+--------+
|2026-06-25 00:00:00|1008   |133.32|android|DE     |/home   |
|2026-06-25 00:05:00|1077   |29.91 |ios    |US     |/search |
|2026-06-25 00:10:00|1065   |209.2 |web    |CA     |/product|
|2026-06-25 00:15:00|1043   |17.73 |web    |US     |/home   |
|2026-06-25 00:20:00|1043   |231.59|web    |CA     |/search |
+-------------------+-------+------+-------+-------+--------+
only showing top 5 rows



In [25]:
from pyspark.sql import functions as F

events_sdf = (
    events_sdf
    .withColumn("event_time", F.to_timestamp("event_time"))
    .select("event_time", "user_id", "amount", "device", "country", "page_url")
)

events_sdf.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- page_url: string (nullable = true)



In [26]:
spark.sql("""
CREATE TABLE local.demo.events_iceberg (
    event_time TIMESTAMP,
    user_id BIGINT,
    amount DOUBLE,
    device STRING,
    country STRING,
    page_url STRING
)
USING iceberg
PARTITIONED BY (days(event_time))
TBLPROPERTIES (
    'format-version' = '2',
    'write.object-storage.enabled' = 'true',
    'write.object-storage.partitioned-paths' = 'false',
    'write.target-file-size-bytes' = '524288'
)
""")

DataFrame[]

In [27]:
events_sdf.createOrReplaceTempView("events_csv")

# 2026-06-25, first half
spark.sql("""
INSERT INTO local.demo.events_iceberg
SELECT *
FROM events_csv
WHERE event_time >= TIMESTAMP '2026-06-25 00:00:00'
  AND event_time <  TIMESTAMP '2026-06-25 12:00:00'
""")

# 2026-06-25, second half
spark.sql("""
INSERT INTO local.demo.events_iceberg
SELECT *
FROM events_csv
WHERE event_time >= TIMESTAMP '2026-06-25 12:00:00'
  AND event_time <  TIMESTAMP '2026-06-26 00:00:00'
""")

# 2026-06-26, early
spark.sql("""
INSERT INTO local.demo.events_iceberg
SELECT *
FROM events_csv
WHERE event_time >= TIMESTAMP '2026-06-26 00:00:00'
  AND event_time <  TIMESTAMP '2026-06-26 08:00:00'
""")

# 2026-06-26, middle
spark.sql("""
INSERT INTO local.demo.events_iceberg
SELECT *
FROM events_csv
WHERE event_time >= TIMESTAMP '2026-06-26 08:00:00'
  AND event_time <  TIMESTAMP '2026-06-26 16:00:00'
""")

# 2026-06-26, late
spark.sql("""
INSERT INTO local.demo.events_iceberg
SELECT *
FROM events_csv
WHERE event_time >= TIMESTAMP '2026-06-26 16:00:00'
  AND event_time <  TIMESTAMP '2026-06-27 00:00:00'
""")

DataFrame[]

In [28]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM local.demo.events_iceberg
""").show()

+---------+
|row_count|
+---------+
|      576|
+---------+



In [31]:
table_path = iceberg_warehouse / "demo" / "events_iceberg"


for path in sorted(table_path.rglob("*")):
    print(path.relative_to(table_path))

data
data/0000
data/0000/1111
data/0000/1111/1010
data/0000/1111/1010/.01010011-00000-11-19d6da8d-0a1a-41bc-9dc1-8938f5e2f8e9-0-00001.parquet.crc
data/0000/1111/1010/01010011-00000-11-19d6da8d-0a1a-41bc-9dc1-8938f5e2f8e9-0-00001.parquet
data/0010
data/0010/0001
data/0010/0001/0010
data/0010/0001/0010/.01100000-00000-13-b295879d-c56d-423a-aa93-6fa07fcc9d91-0-00001.parquet.crc
data/0010/0001/0010/01100000-00000-13-b295879d-c56d-423a-aa93-6fa07fcc9d91-0-00001.parquet
data/0100
data/0100/1100
data/0100/1100/0100
data/0100/1100/0100/.00010111-00000-7-24799120-2cb1-4870-945f-cec7b5185661-0-00001.parquet.crc
data/0100/1100/0100/00010111-00000-7-24799120-2cb1-4870-945f-cec7b5185661-0-00001.parquet
data/1010
data/1010/1110
data/1010/1110/1001
data/1010/1110/1001/.10000011-00000-15-3f9770bd-6c75-439d-b66c-a5df3be71095-0-00001.parquet.crc
data/1010/1110/1001/10000011-00000-15-3f9770bd-6c75-439d-b66c-a5df3be71095-0-00001.parquet
data/1100
data/1100/0011
data/1100/0011/0100
data/1100/0011/0100/.101

In [32]:
spark.sql("""
SELECT event_time, user_id, amount
FROM local.demo.events_iceberg
WHERE event_time >= TIMESTAMP '2026-06-26 10:15:00'
  AND event_time <  TIMESTAMP '2026-06-26 10:30:00'
ORDER BY event_time
""").show(truncate=False)

+-------------------+-------+------+
|event_time         |user_id|amount|
+-------------------+-------+------+
|2026-06-26 10:15:00|1022   |78.35 |
|2026-06-26 10:20:00|1027   |64.14 |
|2026-06-26 10:25:00|1013   |84.01 |
+-------------------+-------+------+



In [33]:
spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    parent_id,
    operation,
    manifest_list
FROM local.demo.events_iceberg.snapshots
ORDER BY committed_at
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                                          |
+-----------------------+-------------------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|2026-06-26 22:49:23.519|6648816474798334786|NULL               |append   |/opt/airflow/data/iceberg_hive_demo/iceberg_warehouse/demo/events_iceberg/metadata/snap-6648816474798334786-1-9a4ea5f4-1936-4891-8e44-bf204233dbf2.avro|
|2026-06-26 22:49:25.257|8696993581391200588|6648816474798334786|append   |/opt/airflow/

In [34]:
spark.sql("""
SELECT
    path,
    length,
    partition_spec_id,
    added_snapshot_id,
    added_data_files_count,
    existing_data_files_count,
    deleted_data_files_count,
    partition_summaries
FROM local.demo.events_iceberg.manifests
ORDER BY path
""").show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+----------------------------------------+
|path                                                                                                                           |length|partition_spec_id|added_snapshot_id  |added_data_files_count|existing_data_files_count|deleted_data_files_count|partition_summaries                     |
+-------------------------------------------------------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+----------------------------------------+
|/opt/airflow/data/iceberg_hive_demo/iceberg_warehouse/demo/events_iceberg/metadata/13e834ab-9d39-4451-9c5f-28f3d6e33232-m0.avro|7

In [35]:
spark.sql("""
SELECT
    file_path,
    partition,
    record_count,
    file_size_in_bytes
FROM local.demo.events_iceberg.files
ORDER BY file_path
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+------------+------------------+
|file_path                                                                                                                                                           |partition   |record_count|file_size_in_bytes|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+------------+------------------+
|/opt/airflow/data/iceberg_hive_demo/iceberg_warehouse/demo/events_iceberg/data/0000/1111/1010/01010011-00000-11-19d6da8d-0a1a-41bc-9dc1-8938f5e2f8e9-0-00001.parquet|{2026-06-26}|96          |3104              |
|/opt/airflow/data/iceberg_hive_demo/iceberg_warehouse/demo/events_iceberg/data/0010/0001/0010/01100000-00000-13-b295879d-c56d-423a-aa93-6fa07fcc9d91-0-

In [36]:
spark.sql("""
SELECT *
FROM local.demo.events_iceberg.partitions
ORDER BY partition
""").show(truncate=False)

+------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition   |spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{2026-06-25}|0      |288         |2         |7021                         |0                           |0                         |0                           |0                         |2026-06-26 22:49:25.257|8696993581391200588     |
|{2026-06-26}|0      |288         |3         |93

In [37]:
files_df = spark.sql("""
SELECT
    file_path,
    partition,
    record_count,
    lower_bounds,
    upper_bounds,
    value_counts,
    null_value_counts
FROM local.demo.events_iceberg.files
ORDER BY file_path
""")

files_df.show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------+
|file_path                                                                                                                                                           |partition   |record_count|lower_bounds                                                                                                                                                      |upper_bounds          

In [38]:
import struct
from pyspark.sql.functions import udf
from pyspark.sql.types import MapType, IntegerType, StringType

# 1. Define a robust decoder that dynamically handles data types based on byte length
def safe_decode_binary_map(binary_map):
    if not binary_map:
        return {}
        
    decoded_map = {}
    for col_id, b_val in binary_map.items():
        if b_val is None or len(b_val) == 0:
            decoded_map[col_id] = "None"
            continue
            
        byte_len = len(b_val)
        bytes_data = bytes(b_val)
        
        try:
            if byte_len == 1:
                # 1 byte: Boolean or tinyint
                decoded_map[col_id] = str(struct.unpack('?', bytes_data)[0])
            elif byte_len == 2:
                # 2 bytes: Short integer
                decoded_map[col_id] = str(struct.unpack('<h', bytes_data)[0])
            elif byte_len == 4:
                # 4 bytes: Integer or Float
                # Try integer parsing first, fall back to float if it looks strange
                val = struct.unpack('<i', bytes_data)[0]
                # If the integer is excessively massive, it might be a float
                if abs(val) > 10000000:
                    decoded_map[col_id] = f"{struct.unpack('<f', bytes_data)[0]:.4f}"
                else:
                    decoded_map[col_id] = str(val)
            elif byte_len == 8:
                # 8 bytes: Long integer, BigInt, or Double (very common for timestamps/metrics)
                try:
                    # Let's try parsing as a Double float first
                    decoded_map[col_id] = f"{struct.unpack('<d', bytes_data)[0]:.4f}"
                except:
                    # Fallback to a Long integer
                    decoded_map[col_id] = str(struct.unpack('<q', bytes_data)[0])
            else:
                # Strings or larger structures: Decode directly as UTF-8 string characters
                decoded_map[col_id] = bytes_data.decode('utf-8', errors='replace')
        except Exception:
            # Fallback wrapper if decoding fails to ensure the query never crashes
            decoded_map[col_id] = f"0x{bytes_data.hex()}"
            
    return decoded_map

# 2. Register the UDF with String values so all mixed types can be displayed side-by-side cleanly
decode_map_udf = udf(safe_decode_binary_map, MapType(IntegerType(), StringType()))

# 3. Read the metadata table exactly like your original query
raw_files_df = spark.sql("""
    SELECT 
        substring_index(file_path, '/', -1) AS file_name, -- Shortens path to fit screen
        partition,
        record_count,
        lower_bounds,
        upper_bounds,
        value_counts,
        null_value_counts
    FROM local.demo.events_iceberg.files
""")

# 4. Clean up the min/max bounds columns using the safe dynamic UDF
formatted_df = raw_files_df \
    .withColumn("lower_bounds", decode_map_udf("lower_bounds")) \
    .withColumn("upper_bounds", decode_map_udf("upper_bounds")) \
    .orderBy("file_name")

# 5. Output cleanly as a flawless DataFrame grid in your Jupyter Notebook
formatted_df.toPandas()

,file_name,partition,record_count,lower_bounds,upper_bounds,value_counts,null_value_counts
0,00010111-00000-7-24799120-2cb1-4870-945f-cec7b5185661-0-00001.parquet,"(2026-06-25,)",144,"{1: '0.0000', 2: '0.0000', 3: '6.3300', 4: 'android', 5: '16707', 6: '/cart'}","{1: '0.0000', 2: '0.0000', 3: '249.7800', 4: 'web', 5: '21333', 6: '/search'}","{1: 144, 2: 144, 3: 144, 4: 144, 5: 144, 6: 144}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0}"
1,01010011-00000-11-19d6da8d-0a1a-41bc-9dc1-8938f5e2f8e9-0-00001.parquet,"(2026-06-26,)",96,"{1: '0.0000', 2: '0.0000', 3: '6.1000', 4: 'android', 5: '16707', 6: '/cart'}","{1: '0.0000', 2: '0.0000', 3: '249.6900', 4: 'web', 5: '21333', 6: '/search'}","{1: 96, 2: 96, 3: 96, 4: 96, 5: 96, 6: 96}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0}"
2,01100000-00000-13-b295879d-c56d-423a-aa93-6fa07fcc9d91-0-00001.parquet,"(2026-06-26,)",96,"{1: '0.0000', 2: '0.0000', 3: '5.3000', 4: 'android', 5: '16707', 6: '/cart'}","{1: '0.0000', 2: '0.0000', 3: '249.4100', 4: 'web', 5: '21333', 6: '/search'}","{1: 96, 2: 96, 3: 96, 4: 96, 5: 96, 6: 96}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0}"
3,10000011-00000-15-3f9770bd-6c75-439d-b66c-a5df3be71095-0-00001.parquet,"(2026-06-26,)",96,"{1: '0.0000', 2: '0.0000', 3: '5.9800', 4: 'android', 5: '16707', 6: '/cart'}","{1: '0.0000', 2: '0.0000', 3: '248.3100', 4: 'web', 5: '21333', 6: '/search'}","{1: 96, 2: 96, 3: 96, 4: 96, 5: 96, 6: 96}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0}"
4,10110111-00000-9-89d0af07-ae5a-4f2c-a4b1-31ca75a34a95-0-00001.parquet,"(2026-06-25,)",144,"{1: '0.0000', 2: '0.0000', 3: '7.6300', 4: 'android', 5: '16707', 6: '/cart'}","{1: '0.0000', 2: '0.0000', 3: '248.8900', 4: 'web', 5: '21333', 6: '/search'}","{1: 144, 2: 144, 3: 144, 4: 144, 5: 144, 6: 144}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0}"


In [39]:
spark.sql("""
DESCRIBE TABLE EXTENDED local.demo.events_iceberg
""").show(100, truncate=False)

+-----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                |comment|
+-----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|event_time                   |timestamp                                                                                                                                          

In [41]:
print(spark.sql("""
EXPLAIN EXTENDED
SELECT event_time, user_id, amount
FROM local.demo.events_iceberg
WHERE event_time >= TIMESTAMP '2026-06-26 10:15:00'
  AND event_time <  TIMESTAMP '2026-06-26 10:30:00'
""").show(truncate=False))

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [44]:
# Stop Spark when finished.
spark.stop()

import shutil
shutil.rmtree(base_dir)

FileNotFoundError: [Errno 2] No such file or directory: '/opt/airflow/data/iceberg_hive_demo'